# 11 · Stacker with token-tagger features

The same stacker as notebook 09, with one change: the eight per-text summaries of the token tagger (notebook 10) are
added as a new block. The tagger marks which tokens an AI wrote, so a fusion text should show one AI section inside
human text, while an ai text is AI everywhere and a human text nowhere. The reference is the best stacker of
notebook 09 (gradient boosting with all features, 0.964 out-of-fold, 0.964 on the public leaderboard); all scores are
out-of-fold on the same 5 group folds inside validation. The tagger was trained on the training part only, so its
features on validation are honest.

In [1]:
# Setup: clone/update the repo on Colab/Kaggle, install requirements, resolve paths, get the data
import os, sys, subprocess
REPO_URL = "https://github.com/anna-kryvulia/ucu-ai-hw-generated-text-detection.git"
BRANCH = "main"   # branch with the code you pushed
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    ROOT = "/content/repo" if "google.colab" in sys.modules else "/kaggle/working/repo"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, f"{ROOT}/src")

from hwkit import env
from hwkit.download import ensure_data
from hwkit.tracker import Run
P = env.setup(seed=42)
ensure_data(P)

platform: local
    data: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data
 outputs: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/outputs
   cache: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/outputs/cache
    repo: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection
  device: mps
data ready at /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data


PosixPath('/Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data')

In [2]:
# Imports, config; validation and test texts; features of every block for both sets
import json
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from hwkit.config import load_config
from hwkit.split import LABELS, load_split, scores
from hwkit.stack import chunk_features, group_folds, out_of_fold, text_features
from hwkit.submission import make_submission

NOTEBOOK_START = time.time()
cfg = load_config("stack_tagger")
METRICS = ["macro_f1_testlike", "macro_f1", "accuracy", "f1_human", "f1_fusion", "f1_ai"]

val = load_split(pd.read_csv(P.data / "train.csv", usecols=["id", "text", "label", "domain", "lang"]))
val = val[val["fold"] == 0].reset_index(drop=True)
test = pd.read_csv(P.data / "test.csv")
for df in (val, test):
    df["domain_lang"] = df["domain"] + " | " + df["lang"]
y = val["label"].map({c: i for i, c in enumerate(LABELS)}).to_numpy()
w = val["weight"].to_numpy()
folds = group_folds(val, cfg["cv"]["folds"])
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(val[["domain_lang"]])

def run_features(run_id, split, ids):
    """Chunk features of a saved encoder run, rows in the order of `ids`."""
    pred_dir = P.runs / run_id / "predictions"
    order = pd.read_csv(pred_dir / f"{split}.csv", usecols=["id"])["id"]
    f = chunk_features(dict(np.load(pred_dir / f"chunks_{split}.npz")), len(order), run_id)
    return f.set_index(order).loc[ids].reset_index(drop=True)

def blocks_for(split, df):
    tagger = pd.read_csv(P.runs / cfg["inputs"]["tagger"] / "predictions" / f"features_{split}.csv").set_index("id")
    tfidf = pd.read_csv(P.runs / cfg["inputs"]["baseline"] / "predictions" / f"{split}.csv").set_index("id")
    enc = pd.concat([run_features(r, split, df["id"]) for r in cfg["inputs"]["encoders"]], axis=1)
    base_cols = [c for c in enc.columns if "_mean_" in c or "_first_" in c or c.endswith("_log_chunks")]
    return {
        "base": pd.concat([enc[base_cols],
                           np.log(np.clip(tfidf.loc[df["id"], [f"s_{c}" for c in LABELS]], 1e-6, 1))
                           .reset_index(drop=True).add_prefix("tfidf_"),
                           pd.DataFrame(ohe.transform(df[["domain_lang"]]), columns=ohe.categories_[0])], axis=1),
        "shape": enc.drop(columns=base_cols),
        "text": text_features(df["text"]),
        "tagger": tagger.loc[df["id"]].reset_index(drop=True),
        "domain": pd.DataFrame(ohe.transform(df[["domain_lang"]]), columns=ohe.categories_[0]),
    }

t0 = time.time()
B = {"val": blocks_for("val", val), "test": blocks_for("test", test)}
run = Run(cfg["name"], cfg, P)
print({k: v.shape[1] for k, v in B["val"].items()}, f"features; {(time.time() - t0) / 60:.1f} min")

run 'stack_tagger' at /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/outputs/runs/stack_tagger  done: nothing yet
{'base': 26, 'shape': 10, 'text': 10, 'tagger': 8, 'domain': 9} features; 0.1 min


In [3]:
# Tagger features: 10th, 50th and 90th percentile per class on validation (unweighted)
B["val"]["tagger"].assign(label=val["label"]).groupby("label").quantile([0.1, 0.5, 0.9]).unstack().T.unstack()[LABELS].round(3)

label              human               fusion                   ai         \
                     0.1    0.5    0.9    0.1    0.5    0.9    0.1    0.5   
tag_mean           0.000  0.001  0.037  0.044  0.241  0.525  0.999  1.000   
tag_share          0.000  0.000  0.016  0.040  0.241  0.527  1.000  1.000   
tag_std            0.000  0.003  0.110  0.171  0.414  0.490  0.000  0.000   
tag_runs           0.000  0.000  1.000  1.000  1.000  4.000  1.000  1.000   
tag_longest_share  0.000  0.000  0.013  0.036  0.227  0.516  1.000  1.000   
tag_longest_log    0.000  0.000  2.398  2.639  4.934  5.933  5.220  6.529   
tag_smooth_max     0.001  0.009  0.365  0.407  0.999  1.000  1.000  1.000   
tag_smooth_min     0.000  0.000  0.001  0.000  0.000  0.001  0.994  0.999   

label                     
                     0.9  
tag_mean           1.000  
tag_share          1.000  
tag_std            0.003  
tag_runs           1.000  
tag_longest_share  1.000  
tag_longest_log    7.196  
tag_smooth_max     1.000  
tag_smooth_min     1.000

Only 10% of fusion texts have a smoothed maximum below 0.41, and only 10% of human texts are above 0.37, so most texts fall clearly on one side. The overlap is in fusion texts with a very short AI section and in human texts where the tagger fires on a few tokens.

## Variants, each against the reference

In [4]:
# Out-of-fold scores of each feature set and model; every variant is then refitted on all of validation for test
def logreg(scale=True):
    lr = LogisticRegression(C=cfg["logreg_C"], max_iter=5000)
    return make_pipeline(StandardScaler(), lr) if scale else lr

def boosting():
    return HistGradientBoostingClassifier(**cfg["boosting"], early_stopping=False, random_state=42)

def fit_model(make, X, rows):
    m = make()
    key = f"{m.steps[-1][0]}__sample_weight" if hasattr(m, "steps") else "sample_weight"
    return m.fit(X[rows], y[rows], **{key: w[rows]})

VARIANTS = {                                     # name: (feature blocks, model)
    "tagger features and domain only": (["tagger", "domain"], boosting),
    "all features + tagger": (["base", "shape", "text", "tagger"], boosting),
}
rows, oof, test_prob = [], {}, {}
for name, (blocks, make) in VARIANTS.items():
    t0 = time.time()
    Xv = pd.concat([B["val"][b] for b in blocks], axis=1).to_numpy()
    Xt = pd.concat([B["test"][b] for b in blocks], axis=1).to_numpy()
    oof[name] = out_of_fold(lambda r: fit_model(make, Xv, r), lambda m, r: m.predict_proba(Xv[r]), folds)
    test_prob[name] = fit_model(make, Xv, np.ones(len(val), dtype=bool)).predict_proba(Xt)
    res = scores(val["label"], np.array(LABELS)[oof[name].argmax(axis=1)], val["weight"])
    run.log_metrics(name, res)
    rows.append({"variant": name, **{k: res[k] for k in METRICS}, "minutes": (time.time() - t0) / 60})
ref = json.loads((P.runs / "stack_full" / "metrics.json").read_text())["gradient boosting, all features"]
summary = pd.DataFrame([{"variant": "reference: notebook 09, all features", **{k: ref[k] for k in METRICS}}] + rows)
summary.insert(1, "vs reference", summary["macro_f1_testlike"] - summary.loc[0, "macro_f1_testlike"])
summary.round(4)

,variant,vs reference,macro_f1_testlike,macro_f1,accuracy,f1_human,f1_fusion,f1_ai,minutes
0,"reference: notebook 09, all features",0.0000,0.9642,0.9664,0.9720,0.9463,0.9554,0.9975,NaN
1,tagger features and domain only,-0.0072,0.9570,0.9592,0.9656,0.9367,0.9460,0.9950,0.3352
2,all features + tagger,0.0030,0.9672,0.9692,0.9744,0.9503,0.9596,0.9977,0.4312


The tagger features with domain only reach 0.957, which is almost the full stacker of notebook 09 (0.964) from 17 features. Added to all other features they give 0.967 (+0.003). The gain is in human (+0.004) and fusion (+0.004); ai was already at 0.998. On the public leaderboard this submission scores 0.968 (notebook 09: 0.964).

In [5]:
# Macro-F1 per domain (over the classes present there): reference (saved out-of-fold predictions) vs the best variant
best = summary.iloc[1:].sort_values("macro_f1_testlike").iloc[-1]["variant"]
ref_oof = pd.read_csv(P.runs / "stack_full" / "predictions" / "val_oof.csv").set_index("id")
oof["reference"] = ref_oof.loc[val["id"], [f"s_{c}" for c in LABELS]].to_numpy()
dom = pd.DataFrame({name: {dl: f1_score(g["label"], np.array(LABELS)[oof[name][g.index].argmax(axis=1)],
                                        labels=sorted(g["label"].unique()), average="macro", zero_division=0)
                           for dl, g in val.groupby("domain_lang")}
                    for name in ("reference", best)})
dom.columns = ["reference", "best"]
dom["texts"] = val["domain_lang"].value_counts()
dom["change"] = dom["best"] - dom["reference"]
print("best variant:", best)
dom.round(3).sort_values("change")

best variant: all features + tagger


,reference,best,texts,change
bundestag | de-DE,0.972,0.974,5234,0.002
cnn_news | en-EN,0.991,0.993,4429,0.002
euro_court_cases | en-EN,0.977,0.980,2759,0.003
student_essays | en-EN,0.973,0.975,5411,0.003
arxiv_papers | en-EN,0.965,0.969,1429,0.004
blog_authorship_corpus | en-EN,0.911,0.915,4053,0.004
house_of_commons | en-EN,0.975,0.979,3186,0.004
gutenberg | en-EN,0.945,0.956,692,0.011
gutenberg | de-DE,0.806,0.905,43,0.099


Every domain improves. The largest change is German Gutenberg (+0.10), but it has only 43 validation texts. Blogs stay the weakest domain (0.915).

In [6]:
# Confusion matrix of the best variant on validation (out-of-fold, rows = true label, counts)
pd.crosstab(val["label"], pd.Series(np.array(LABELS)[oof[best].argmax(axis=1)], name="predicted"))[LABELS].loc[LABELS]

predicted,human,fusion,ai
label,,,
human,6421,239,14
fusion,403,7948,16
ai,15,11,12169


Almost all remaining errors are between human and fusion: 403 fusion texts are called human and 239 human texts are called fusion. Only 56 errors involve the ai class.

In [7]:
# Save out-of-fold and test probabilities of the best variant; write its submission file
cols = [f"s_{c}" for c in LABELS]
pd.concat([val[["id", "label"]], pd.DataFrame(oof[best], columns=cols)], axis=1).to_csv(run.pred_path("val_oof"), index=False)
pd.concat([test[["id"]], pd.DataFrame(test_prob[best], columns=cols)], axis=1).to_csv(run.pred_path("test"), index=False)
sub = make_submission(test["id"], np.array(LABELS)[test_prob[best].argmax(axis=1)],
                      P.repo / "submissions" / f"{cfg['name']}.csv", P.data)

submission saved: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/submissions/stack_tagger.csv  (25000 rows)
label
ai        0.513
human     0.259
fusion    0.228


In [8]:
# Distinct warnings logged in this session (should be empty), total time
print(f"total: {(time.time() - NOTEBOOK_START) / 60:.1f} min")
env.warnings_summary(P)

total: 1.1 min


[]